In [3]:
from google.colab import drive
drive.mount('/content/drive')

!ls -lh /content/drive/MyDrive | grep -i archive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root  65M Oct  3 12:08 archive.zip


In [4]:
!find /content/drive/MyDrive -name "archive.zip"

/content/drive/MyDrive/archive.zip


In [5]:
!cp /content/drive/MyDrive/archive.zip /content/
!unzip -q /content/archive.zip -d /content/data

In [6]:
!ls /content/data

test  train


In [7]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
!ls /content/data/train
!find /content/data/train -type f | wc -l

cats  dogs
557


In [9]:
IMG_SIZE = (160, 160)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/data/train', validation_split=0.2, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/data/train', validation_split=0.2, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH)

test_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/data/test', image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)

class_names = train_ds.class_names
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Found 557 files belonging to 2 classes.
Using 446 files for training.
Found 557 files belonging to 2 classes.
Using 111 files for validation.
Found 140 files belonging to 2 classes.
['cats', 'dogs']


In [11]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [12]:
base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False   # freeze the pretrained part

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1)(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [15]:
from google.colab import drive
drive.mount('/content/drive')

ckpt = tf.keras.callbacks.ModelCheckpoint(
    '/content/drive/MyDrive/cat_dog_best.keras',
    save_best_only=True, monitor='val_accuracy')

history = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[ckpt])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 14s 260ms/step - accuracy: 0.6771 - loss: 0.5623 - val_accuracy: 0.7568 - val_loss: 0.3614
Epoch 2/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 195ms/step - accuracy: 0.8386 - loss: 0.3460 - val_accuracy: 0.8559 - val_loss: 0.2473
Epoch 3/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 277ms/step - accuracy: 0.8700 - loss: 0.2583 - val_accuracy: 0.8559 - val_loss: 0.2206
Epoch 4/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 181ms/step - accuracy: 0.9103 - loss: 0.2244 - val_accuracy: 0.8919 - val_loss: 0.1881
Epoch 5/5
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 165ms/step - accuracy: 0.9193 - loss: 0.1915 - val_accuracy: 0.9189 - val_loss: 0.1693


In [16]:
loss, acc = model.evaluate(test_ds)
print(f"Test accuracy: {acc:.3f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 218ms/step - accuracy: 0.8714 - loss: 0.2333
Test accuracy: 0.871


In [17]:
import numpy as np

def predict(path):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[np.newaxis]
    p = float(tf.sigmoid(model.predict(arr, verbose=0))[0][0])
    label = class_names[1] if p > 0.5 else class_names[0]
    conf = p if p > 0.5 else 1 - p
    print(f"{path}: {label} ({conf:.1%} confident)")

predict('/content/drive/MyDrive/cat.jpg')
predict('/content/drive/MyDrive/dog.jpg')

/content/drive/MyDrive/cat.jpg: cats (56.2% confident)
/content/drive/MyDrive/dog.jpg: dogs (97.2% confident)


In [24]:
model.save('/content/drive/MyDrive/cat_dog_model.keras')

In [27]:
print(tf.__version__)
print(class_names)

2.20.0
['cats', 'dogs']
